# Initial Data Inspection of Fridays For Future Insta Postings
***


## 1) Load Data

In [ ]:
import torch
torch.cuda.empty_cache()
print(torch.__version__)
torch.cuda.is_available()

In [ ]:
import sys
from pathlib import Path
import sshfs

sys.path.append(str(Path("..").resolve()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src import plot_style
import json
import re
from collections import Counter
import yaml 
import torch
from spacy.lang.de.stop_words import STOP_WORDS as DE_STOP_WORDS
from sklearn.feature_extraction.text import CountVectorizer
from sentence_transformers import SentenceTransformer
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN
from src.crisis_windows import CRISIS_WINDOWS, resolve_windows, filter_by_window


# data server connection
#with open("C:/Users/Elena Solar/.ssh/sshfs_config.yaml") as f:
#    config = yaml.safe_load(f)
with open("../../login/sshfs_config.yaml") as f:
    config = yaml.safe_load(f)

fs = sshfs.SSHFileSystem(
    host=config["sshfs"]["host"],
    username=config["sshfs"]["username"],
    client_keys=config["sshfs"]["client_keys"],
)



plot_style.set_style()

In [ ]:
# Read and combine files

with fs.open("/data/workspace/esolar/FFF_Climate_Framing/data/raw/FFF_german_01-2015_12-2022.csv") as f:
    df_1 = pd.read_csv(f, low_memory = False)

with fs.open("/data/workspace/esolar/FFF_Climate_Framing/data/raw/FFF_german_01-2023_07-2026.csv") as f:
    df_2 = pd.read_csv(f, low_memory = False)

df = pd.concat([df_1, df_2]).sort_values(by="creation_time").reset_index(drop=True)
df["creation_time"] = pd.to_datetime(df["creation_time"])
df

### Data Frame Columns

* `content_type`: type of post, i.e. ['videos', 'albums', 'photos', 'stories']
* `creation_time`: timestamp the post was published
* `hashtags`: list of hashtags used in the post
* `id`: unique identifier of the post
* `is_branded_content`: whether the post is flagged as branded [False/True]
* `lang`: detected language of the post text
* `match_type`: how the post matched the search query/criteria, ['["post_text"]', '["multimedia_text"]', nan]
* `mcl_url`: URL to the post in the Meta Content Library
* `modified_time`: timestamp the post was last modified
* `multimedia`: attached media objects (images/videos) and their metadata
* `post_owner.id`: unique identifier of the posting account
* `post_owner.name`: display name of the posting account
* `post_owner.type`: account type, i.e. ['creator', 'business', 'personal']
* `post_owner.username`: handle/username of the posting account
* `statistics.comment_count`: number of comments on the post
* `statistics.like_count`: number of likes on the post
* `statistics.views`: number of views
* `statistics.views_date_last_refreshed`: timestamp the views count was last updated
* `text`: post caption/body text

---
## 2) Descriptive Statistics

### Content Type

In [ ]:
content_type_counts = df["content_type"].value_counts()
content_type_pct = (content_type_counts / content_type_counts.sum() * 100).round(1)

fig, ax = plt.subplots(figsize=plot_style.FIGSIZE_SINGLE)
sns.countplot(data=df, x="content_type", order=content_type_counts.index, ax=ax)

# annotate with count (share%) instead of just count
for patch, count, pct in zip(ax.patches, content_type_counts, content_type_pct):
    ax.annotate(f"{count} ({pct}%)",
                (patch.get_x() + patch.get_width() / 2, patch.get_height()),
                ha="center", va="bottom", fontsize=9, xytext=(0, 2),
                textcoords="offset points")

ax.set_title("Posts by Content Type")
ax.set_xlabel("Content Type")
ax.set_ylabel("Number of Posts")
#plot_style.savefig(fig, "content_type_counts")
plt.show()


### Posting Volume Over Time

In [ ]:
daily_by_type = (
    df.groupby([df["creation_time"].dt.floor("D"), "content_type"])
    .size()
    .unstack(fill_value=0)
    .rolling(30).mean()
)
daily_total = daily_by_type.sum(axis=1)

daily_by_type_long = daily_by_type.reset_index().melt(
    id_vars="creation_time", var_name="content_type", value_name="posts_per_day"
)

fig, ax = plt.subplots(figsize=plot_style.FIGSIZE_WIDE)
sns.lineplot(
    data=daily_by_type_long, x="creation_time", y="posts_per_day",
    hue="content_type", linewidth=1, alpha=0.85, ax=ax,
)
ax.plot(daily_total.index, daily_total, color="black", linewidth=1.8, label="All posts")

ax.set_title("Post Volume Over Time (30-day rolling average)")
#ax.set_xlim(2018, 2026)
ax.set_xlabel("Date")
ax.set_ylabel("Posts per Day")
ax.legend(title="Content Type", loc="upper right")
plot_style.savefig(fig, "post_volume_timeline")
plt.show()

### Account Type

In [ ]:
owner_type_counts = df["post_owner.type"].value_counts()

fig, ax = plt.subplots(figsize=plot_style.FIGSIZE_SINGLE)
sns.countplot(data=df, x="post_owner.type", order=owner_type_counts.index, ax=ax)
plot_style.annotate_bars(ax)
ax.set_title("Posts by Account Type")
ax.set_xlabel("Account Type")
ax.set_ylabel("Number of Posts")
#plot_style.savefig(fig, "account_type_counts")
plt.show()

### Engagement: Likes & Views

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=plot_style.FIGSIZE_WIDE)
seq_colors = sns.color_palette(plot_style.PALETTE_SEQUENTIAL, 6)

sns.histplot(df["statistics.like_count"].dropna(), bins=50, ax=axes[0], color=seq_colors[2])
axes[0].set_yscale("log")
axes[0].set_title("Distribution of Like Counts")
axes[0].set_xlabel("Likes")

sns.histplot(df["statistics.views"].dropna(), bins=50, ax=axes[1], color=seq_colors[4])
axes[1].set_yscale("log")
axes[1].set_title("Distribution of View Counts")
axes[1].set_xlabel("Views")

plt.tight_layout()
#plot_style.savefig(fig, "likes_views_distributions")
plt.show()

### Language

In [ ]:
lang_counts = df["lang"].value_counts().head(15)

fig, ax = plt.subplots(figsize=plot_style.FIGSIZE_TALL)
sns.countplot(
    data=df[df["lang"].isin(lang_counts.index)], y="lang", order=lang_counts.index, ax=ax,
)
plot_style.annotate_bars(ax)
ax.set_title("Top 15 Languages by Post Count")
ax.set_xlabel("Number of Posts")
ax.set_ylabel("Language")
#plot_style.savefig(fig, "language_counts")
plt.show()

### Top Accounts

**Decision** 

Exclude the account of `hackhermannjosef`, as he is noise, but dominates the post count as well as hashtag count

In [ ]:
df["post_owner.username"].value_counts().head(20)

In [ ]:
df = df[df["post_owner.username"] != "hackhermannjosef"].reset_index(drop=True)
df["post_owner.username"].value_counts().head(20)

### Official Fridays For Future Accounts

In [ ]:
fff_mask = df["post_owner.username"].str.contains(
    r"fridaysforfuture|fridays_for_future", case=False, na=False, regex=True
)
fff_account_counts = pd.DataFrame(df.loc[fff_mask, "post_owner.username"].value_counts()).reset_index()

print(f"{fff_mask.sum()} posts from {fff_account_counts.shape[0]} accounts matching "
      f"'fridaysforfuture' / 'fridays_for_future'")
fff_account_counts

### Hashtags

In [ ]:
hashtag_lists = df["hashtags"].dropna().apply(json.loads)
hashtag_counts = Counter(tag.lower() for tags in hashtag_lists for tag in tags)

top_hashtags = pd.Series(dict(hashtag_counts.most_common(30)))
total_tags = sum(hashtag_counts.values())
top_hashtags_pct = (top_hashtags / len(df) * 100).round(1)

fig, ax = plt.subplots(figsize=plot_style.FIGSIZE_TALL)
sns.barplot(x=top_hashtags.values, y=top_hashtags.index, order=top_hashtags.index, ax=ax)

# annotate with count (share%) instead of just count
for patch, count, pct in zip(ax.patches, top_hashtags, top_hashtags_pct):
    ax.annotate(f"{count} ({pct}%)",
                (patch.get_width(), patch.get_y() + patch.get_height() / 2),
                ha="left", va="center", fontsize=9, xytext=(4, 0),
                textcoords="offset points")

ax.set_title("Top 30 Hashtags")
ax.set_xlabel("Number of Posts")
ax.set_ylabel("Hashtag")
#plot_style.savefig(fig, "top_hashtags")
plt.show()

## 3) Sample Composition & Quality Checks

### Restrict to FridaysForFuture Accounts


In [ ]:
df_FFF = df[df["post_owner.username"].isin(fff_account_counts["post_owner.username"])].reset_index(drop=True)

df_no_FFF = df[~df["post_owner.username"].isin(
    fff_account_counts["post_owner.username"])].reset_index(drop=True)


In [ ]:
# Scope decision (notes/data_collection.md): Germany-based chapters only -- the crisis
# windows and key events are German (Bundestag elections, Heizungsgesetz, lockdown timing),
# so the Austrian chapters (~20% of FFF-pattern posts) are dropped rather than pooled. The
# analysis period ends with 2025: the pull's 2026 slice (Jan-Jul) has only ~50 posts, too
# few for the quarterly and per-window comparisons downstream.
GERMANY_FFF_ACCOUNTS = [
    "fridaysforfuture.de",
    "fridaysforfuture.berlin",
    "fridaysforfuture.koeln",
    "fridaysforfuturemuenchen",
    "fridaysforfuture_hh",
]
ANALYSIS_END = pd.Timestamp("2025-12-31 23:59:59", tz="UTC")

n_all_fff = len(df_FFF)
df_FFF = df_FFF[
    df_FFF["post_owner.username"].isin(GERMANY_FFF_ACCOUNTS)
    & (pd.to_datetime(df_FFF["creation_time"], utc=True) <= ANALYSIS_END)
].reset_index(drop=True)
print(f"{n_all_fff} posts from all FFF-pattern accounts -> {len(df_FFF)} from Germany-based "
      f"chapters through {ANALYSIS_END.date()}")
print(df_FFF["post_owner.username"].value_counts())

df_FFF.to_csv("../data/processed/FFF_german.csv", index=False)
print("saved base FFF_german.csv -- 00_data_inspection_FFF.ipynb adds clean_text/logistics/topic "
      "columns and saves it back")

### Check Texts

In [ ]:
sample = df["text"].sample(3).reset_index(drop = True)

for i in range(len(sample)):
    print(sample[i])
    print("------------")

### Substantive Text Content

Hashtags and (redacted) mentions inflate raw text length without carrying diagnostic/prognostic/motivational content. Strip both, then check how many posts still have enough words left to plausibly make a statement or mobilize people.

In [ ]:
def strip_hashtags_mentions(text):
    if pd.isna(text):
        return ""
    text = re.sub(r"#\S+", " ", text)              # hashtags
    text = re.sub(r"@\S+", " ", text)               # raw @mentions
    text = re.sub(r"<redacted_mention>", " ", text)  # anonymized mentions
    text = re.sub(r"\n", "", text)                 # remove linebreaks
    text = re.sub(r" +", ' ', text)                   # remove repeated spaces
    return text

df["clean_text"] = df["text"].apply(strip_hashtags_mentions)
df["clean_word_count"] = df["clean_text"].str.split().str.len().fillna(0).astype(int)

n_empty = (df["clean_word_count"] == 0).sum()
print(f"posts with 0 words after stripping hashtags/mentions: {n_empty} "
      f"({n_empty / len(df) * 100:.1f}%)")

for threshold in (10, 20, 30, 40, 50):
    n = (df["clean_word_count"] >= threshold).sum()
    print(f"posts with >= {threshold} words remaining: {n} ({n / len(df) * 100:.1f}%)")

In [ ]:
fig, ax = plt.subplots(figsize=plot_style.FIGSIZE_WIDE)
sns.histplot(df["clean_word_count"], bins=60, ax=ax)
for threshold, style in zip((10, 20, 30), ("--", ":", "--")):
    ax.axvline(threshold, color="black", linestyle=style, linewidth=1, label=f"{threshold} words")

ax.set_xlim(0, 100)
ax.set_title("Word Count per Post (Hashtags & Mentions Stripped)")
ax.set_xlabel("Words")
ax.set_ylabel("Number of Posts")
ax.legend()
#plot_style.savefig(fig, "clean_word_count_distribution")
plt.show()

In [ ]:
list(df[df["clean_word_count"] < 5]["clean_text"])

### Protest Logistics Heuristics

A regex-based flag for event-logistics language (times, dates, "Treffpunkt"-style call-to-action markers) to test directly whether "logistics-heavy" posts are shorter/less substantive than the rest.

In [ ]:

# regex-based logistics markers: clock times, dates, and common German call-to-action phrases
TIME_RE = r"\b\d{1,2}[:.]\d{2}\s*uhr\b|\b\d{1,2}\s*uhr\b"
DATE_RE = r"\b\d{1,2}\.\d{1,2}\.(\d{2,4})?\b"
LOGISTICS_KEYWORDS = [
    "treffpunkt", "kundgebung", "startpunkt", "auftaktkundgebung",
    "wann:", "wo:", "ort:", "route:", "start:",
    "kommt alle", "kommt vorbei", "seid dabei", "sei dabei", "mobilisiert",
]

def has_logistics(text):
    t = str(text).lower()
    if re.search(TIME_RE, t) or re.search(DATE_RE, t):
        return True
    return any(kw in t for kw in LOGISTICS_KEYWORDS)

df["has_logistics"] = df["text"].apply(has_logistics)

n_logistics = df["has_logistics"].sum()
print(f"posts with logistics markers (time/date/call-to-action phrasing): "
      f"{n_logistics} ({n_logistics / len(df) * 100:.1f}%)")
print()
print("median clean word count, with vs. without logistics markers:")
print(df.groupby("has_logistics")["clean_word_count"].median())
print()
print("mean clean word count, with vs. without logistics markers:")
print(df.groupby("has_logistics")["clean_word_count"].mean().round(1))


In [ ]:
fig, ax = plt.subplots(figsize=plot_style.FIGSIZE_WIDE)
sns.boxplot(
    data=df, x="has_logistics", y="clean_word_count", ax=ax,
    showfliers=False,
)
ax.set_xticks([0, 1], ["No logistics markers", "Has logistics markers"])
ax.set_title("Word Count by Presence of Event-Logistics Markers")
ax.set_xlabel("")
ax.set_ylabel("Words (hashtags/mentions stripped)")
#plot_style.savefig(fig, "word_count_by_logistics")
plt.show()

### Hashtag Co-occurrence

Pearson correlation between hashtag indicator variables (binary: present/absent per post) across the top 25 hashtags used by official FFF accounts — a fast, non-ML way to see which sub-themes cluster together before trusting the embedding-based topic model below.

In [ ]:
TOP_N_GENERIC = 20
MAX_DOC_FREQ = 0.5  # exclude near-universal branding tags (e.g. #fridaysforfuture, ~82% of posts) -- too little variance to correlate meaningfully with anything
EXTRA_TAGS = [  # thematic tags of interest even if outside the top-N by raw frequency
    ""
]

hashtag_lists = (
    df["hashtags"].dropna().apply(json.loads).apply(lambda tags: [t.lower() for t in tags])
)
n_posts = len(hashtag_lists)
tag_counts = Counter(tag for tags in hashtag_lists for tag in tags)

generic_top_tags = [tag for tag, c in tag_counts.most_common() if c / n_posts <= MAX_DOC_FREQ][:TOP_N_GENERIC]
extra_present = [tag for tag in EXTRA_TAGS if tag in tag_counts]
top_n_tags = list(dict.fromkeys(generic_top_tags + extra_present))  # dedupe, preserve order

hashtag_indicator = pd.DataFrame(
    {tag: hashtag_lists.apply(lambda tags: tag in tags) for tag in top_n_tags}
)
hashtag_corr = hashtag_indicator.corr()
for i in range(len(hashtag_corr)):
    hashtag_corr.iloc[i, i] = np.nan  # mask self-correlation so it doesn't dominate the color scale

fig, ax = plt.subplots(figsize=(8, 7))
sns.heatmap(
    hashtag_corr, cmap=plot_style.PALETTE_SEQUENTIAL, center=0, square=True,
    linewidths=0.4, cbar_kws={"label": "Correlation (co-occurrence)"}, ax=ax,
)
ax.set_title("Hashtag Co-occurrence (all Accounts)")
#plot_style.savefig(fig, "hashtag_cooccurrence_heatmap")
plt.show()

### Hashtag Frequency Table (for Manual Categorization)

Every hashtag used by official FFF accounts, sorted by frequency, with an empty `category` column to fill in by hand (e.g. `climate`, `mobilization`, `feminism`, `democracy`, `other`) — a lightweight manual codebook you can build at your own pace, independent of how the topic model turns out.

In [ ]:
hashtag_counts_df = pd.DataFrame(tag_counts.most_common(), columns=["hashtag", "count"])
hashtag_counts_df["category"] = ""  # fill in by hand

output_path = Path("../data/processed/all_hashtag_counts.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)
hashtag_counts_df.to_csv(output_path, index=False)

print(f"saved {len(hashtag_counts_df)} hashtags to {output_path}")
hashtag_counts_df.head(10)

In [ ]:
hashtag_cat = pd.read_excel("../data/processed/fff_hashtag_categories.xlsx", header=0)
#hashtag_cat = hashtag_cat.rename(columns={0 : "concept", 1 : "hashtags"})
hashtag_cat

### Applying the Manual Hashtag Categories

`hashtag_cat` above is wide -- one row per category, hashtags for that category comma-joined in one cell. Reshape it to a long `hashtag -> category` lookup, sanity-check it against the actual corpus (any hashtag assigned twice? any frequent corpus hashtag left uncategorized?), then attach categories to every post. This is the manual half of Axis E (topic/subject) in Project_Proposal.md §6.3.1 -- BERTopic supplies the other half in §5) below, and the two get cross-checked once topics are fit.

In [ ]:
hashtag_cat_long = hashtag_cat.copy()
hashtag_cat_long["hashtags"] = hashtag_cat_long["hashtags"].str.split(",")
hashtag_cat_long = hashtag_cat_long.explode("hashtags")
hashtag_cat_long["hashtags"] = hashtag_cat_long["hashtags"].str.strip().str.lower()
hashtag_cat_long = hashtag_cat_long.rename(columns={"hashtags": "hashtag"})

print(f"{len(hashtag_cat_long)} (category, hashtag) pairs across {hashtag_cat_long['concept'].nunique()} categories")

dupes = hashtag_cat_long[hashtag_cat_long.duplicated("hashtag", keep=False)].sort_values("hashtag")
if len(dupes):
    print(f"WARNING: {dupes['hashtag'].nunique()} hashtags assigned to more than one category -- "
          f"resolve by hand before treating this as a clean single-label mapping:")
    display(dupes)

hashtag_to_category = dict(zip(hashtag_cat_long["hashtag"], hashtag_cat_long["concept"]))


In [ ]:
# Which real, frequently-used hashtags in the corpus have no manual category at all?
uncategorized = [tag for tag, _ in tag_counts.most_common() if tag not in hashtag_to_category]
n_uncategorized_occurrences = sum(n for tag, n in tag_counts.most_common() if tag not in hashtag_to_category)
print(f"{len(uncategorized)}/{len(tag_counts)} distinct hashtags have no manual category "
      f"({n_uncategorized_occurrences} tag-post occurrences)")
print("most frequent uncategorized hashtags:")
pd.Series({t: tag_counts[t] for t in uncategorized[:20]})


In [ ]:
# FFF's own branding tags (#fridaysforfuture, #fff, ...) sit on the large majority of posts
# (84.9%, see Top 30 Hashtags above) -- present on nearly everything, so (like stripping hashtags
# before BERTopic embedding) they carry no topic signal. Keep "FFF" and "unclassified" out of the
# *topic* variable; hashtag_categories still keeps them for reference.
def categories_for(tags):
    if not isinstance(tags, list):
        return []
    return sorted({hashtag_to_category[t] for t in tags if t in hashtag_to_category})

df["hashtag_categories"] = hashtag_lists.reindex(df.index).apply(categories_for)
df["hashtag_topics"] = df["hashtag_categories"].apply(
    lambda cats: [c for c in cats if c not in ("FFF", "unclassified")]
)

n_no_topic = (df["hashtag_topics"].str.len() == 0).sum()
print(f"posts with >=1 substantive manual category: {len(df) - n_no_topic} "
      f"({(len(df) - n_no_topic) / len(df) * 100:.1f}%)")
print(f"posts with none (no hashtags, or only FFF-branding/unclassified tags): {n_no_topic} "
      f"({n_no_topic / len(df) * 100:.1f}%)")


In [ ]:
category_counts = pd.Series(Counter(cat for cats in df["hashtag_topics"] for cat in cats)).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=plot_style.FIGSIZE_TALL)
sns.barplot(x=category_counts.values, y=category_counts.index, order=category_counts.index, ax=ax)
plot_style.annotate_bars(ax)
ax.set_title("Posts per Manual Hashtag Category (multi-label)")
ax.set_xlabel("Number of Posts")
ax.set_ylabel("Category")
#plot_style.savefig(fig, "hashtag_category_counts")
plt.show()


In [ ]:
hashtag_cat_long[["concept", "hashtag"]].to_csv("../data/processed/all_hashtag_categories_long.csv", index=False)
df[["id", "hashtag_categories", "hashtag_topics"]].to_csv(
    "../data/processed/all_hashtag_topic_assignment.csv", index=False
)
print("saved long-format category lookup and per-post hashtag-topic assignment")


## 4) Manual Sample Per Crisis Window (Inductive Coding)

Per Project_Proposal.md Section 6.1: a stratified read across crisis windows, before touching any classifier or topic model, to sanity-check whether the theoretical categories (DPM, master frame, Mau arenas) actually show up the way expected -- and to build intuition for what "frame extension" into a given window's crisis actually looks like in FFF's own words. Restricted to posts with more than 20 words after stripping hashtags/mentions (`clean_word_count`, from Section 3's "Substantive Text Content" check above) so the sample is worth reading, not logistics-only announcements.

In [ ]:
N_SAMPLES_PER_WINDOW = 8
MIN_WORDS = 20
RANDOM_STATE = 42

resolved_windows = resolve_windows(CRISIS_WINDOWS, df["creation_time"].max())

for name, start, end, description in resolved_windows:
    window_df = filter_by_window(df, (name, start, end, description))
    eligible = window_df[window_df["clean_word_count"] > MIN_WORDS]
    n = min(N_SAMPLES_PER_WINDOW, len(eligible))
    sample = eligible.sample(n, random_state=RANDOM_STATE)

    print("=" * 78)
    print(f"WINDOW: {name}  ({start.date()} to {end.date()})")
    print(f"{description}")
    print(f"{len(window_df)} posts in window, {len(eligible)} with > {MIN_WORDS} words, showing {n}")
    print("=" * 78)
    for _, row in sample.iterrows():
        print(f"\n[{row['creation_time'].date()}] @{row['post_owner.username']}")
        print(row["text"])
        print("-" * 40)
    print()

## 5) Topic Modelling

**Diagnosis of the first run** (2 topics found, 99% of docs in one cluster labeled `fridaysforfuture_klimastreik_klimaschutz_straße`): the hashtag co-occurrence heatmap above shows why -- `#fridaysforfuture` alone is in 82.3% of posts, and the previous `clean_for_topic_model` kept hashtag *words* in the embedding text (just dropped the `#`). A typical caption ends with a block of 10-15 near-synonymous climate/FFF hashtags; repeating "climate, climate crisis, climate change, climate justice, Fridaysforfuture, Klimastreik, Klimaschutz..." at the end of nearly every post floods the pooled sentence embedding and drowns out whatever the actual prose is distinctively about. Fixes applied below:

1. **Strip hashtags entirely from the embedding text** (not just the `#`) -- cluster on the actual prose, not the boilerplate tag block. (Hashtags remain available separately for the co-occurrence analysis above.)
2. **Smaller, explicit HDBSCAN** (`min_cluster_size=10`, `min_samples=5`) -- the themes you're after (democracy/voting, feminism) are genuinely small in this corpus (e.g. `#demokratie` 66 posts, `#feminismus` 2 posts) and would never survive the previous, more conservative default.
3. **Explicit UMAP with a fixed `random_state`** (`n_neighbors=10`) -- smaller neighborhood size preserves more local structure instead of smoothing everything into one blob, and now results are reproducible across reruns (the previous fit had no seed at all).
4. **`seed_topic_list`** nudging BERTopic toward your named themes (climate substance, protest/mobilization logistics, feminism, democracy/voting) -- semi-supervised guidance is a reasonable choice here given you already have strong theoretical priors about what should be there.
5. **Upgraded embedding model** to `paraphrase-multilingual-mpnet-base-v2` (larger/more accurate than the MiniLM model used when this had to run CPU-only) -- cheap now that you have real GPU headroom.
6. **`max_df=0.4` and bigrams** in the vectorizer -- keeps generic boilerplate out of topic *labels* too, and lets 2-word phrases (e.g. "klimastreik hamburg") show up as more specific labels than single words alone.

None of this guarantees the exact 3-4 themes you expect will fall out cleanly -- treat `min_cluster_size`/`min_samples` as the first things to tune further if topics are still too broad or too fragmented.

In [ ]:
print(f"torch {torch.__version__}, CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"device: {torch.cuda.get_device_name(0)}")

In [ ]:
def clean_for_topic_model(text):
    if pd.isna(text):
        return ""
    text = re.sub(r"<redacted_mention>", " ", text)
    text = re.sub(r"@\S+", " ", text)
    text = re.sub(r"https?://\S+", " ", text)
    text = re.sub(r"#\S+", " ", text)  # drop hashtags entirely -- cluster on prose, not the boilerplate tag block
    return re.sub(r"\s+", " ", text).strip()

topic_docs = df["text"].apply(clean_for_topic_model)
non_empty = topic_docs.str.len() > 0
print(f"{(~non_empty).sum()} empty docs excluded from topic modeling")
topic_docs = topic_docs[non_empty].reset_index(drop=True)
topic_dates = df.loc[non_empty, "creation_time"].reset_index(drop=True)

In [ ]:
# Semi-supervised nudge toward the themes you're actually interested in -- tune/extend these lists
# based on what comes out; BERTopic uses them to bias embeddings, not as hard category labels.
seed_topic_list = [
    ["klimawandel", "erderwärmung", "co2", "emissionen"],
    ["klimakrise", "krise", "energiekrise"],
    ["corona", "covid-19"],
    ["ukraine", "krieg", "palestine", "solidarität", "dontfuelwar"],
    ["kohle", "gas", "hambi", "lützi", "kohleausstieg"],
    ["streik", "demo", "demonstration"],
    ["feminismus", "feministisch", "frauen", "gleichberechtigung", "patriarchat", "gender", "pridemonth", "bipoc"],
    ["demokratie", "wahl", "wählen", "bundestagswahl", "stimme", "rechtsextremismus", "afd", "demokratieverteidigen"],
    ["zukunft"],
    ["mobilität", "öffies", "öpnv"]
    
]

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
embedding_model = SentenceTransformer("paraphrase-multilingual-mpnet-base-v2", device=device)

vectorizer_model = CountVectorizer(
    stop_words=list(DE_STOP_WORDS), min_df=1, max_df=1.0, ngram_range=(1, 2),
)

# min_cluster_size is the primary lever for topic count -- raised from 10 to ~1% of the corpus.
# n_neighbors back to UMAP's own default (was lowered to 10 to fight the old collapse-to-1-topic
# problem, which is now fixed upstream by stripping hashtags before embedding).
umap_model = UMAP(
    n_neighbors=10, n_components=5, min_dist=0.0, metric="cosine", random_state=42,
)
hdbscan_model = HDBSCAN(
    min_cluster_size=10, min_samples=5, metric="euclidean",
    cluster_selection_method="leaf", prediction_data=True, # cluster_selection_method="eom"
)

topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    calculate_probabilities=False,
    verbose=True,
)
topics, _ = topic_model.fit_transform(topic_docs.tolist())

In [ ]:
topic_info = topic_model.get_topic_info()

n_outliers = (pd.Series(topics) == -1).sum()
print(f"outlier topic (-1): {n_outliers} / {len(topics)} docs ({n_outliers / len(topics) * 100:.1f}%)")
print(f"{(topic_info['Topic'] != -1).sum()} coherent topics found")
topic_info.head(30)

**Diagnosis of 108 topics / 56.6% outliers**: `min_cluster_size=10`, `min_samples=5`, and `n_neighbors=10` were deliberately aggressive to escape the *previous* problem (everything collapsing into one cluster because boilerplate hashtags dominated the embeddings). That is now fixed structurally (hashtags stripped before embedding), so these same settings just over-fragment: tiny, hyper-local pockets qualify as their own "topic," and many points do not have enough close neighbors to join any of them, so they are dumped as noise instead.

**`min_cluster_size` is the primary lever for topic count** -- raise it and the topic count drops roughly monotonically, since HDBSCAN requires that many points before a cluster counts as a topic at all. A common starting heuristic is ~1% of the corpus (~40 here for n=4,104).

**Two ways to retune, cheapest first:**
1. **`reduce_topics()`** -- merges the *already-fitted* 108 topics down to a target count by representation similarity. No re-embedding, no GPU work, just a fast reduction step -- good for quickly trying several target counts (10 / 15 / 20 / 25) and seeing which reads best.
2. **Re-fit with different HDBSCAN/UMAP parameters** -- the "proper" fix, more expensive (needs the GPU again), but avoids inheriting whatever oddities the over-fragmented base clustering baked in. Recommended once you have used (1) to get a feel for a sensible target count.

In [ ]:
# Cheap, interactive: merge the already-fitted 108 topics down to a target count.
# No re-embedding / no GPU needed for this cell -- just retry with different TARGET_N_TOPICS values.
TARGET_N_TOPICS = 30

topic_model.reduce_topics(topic_docs.tolist(), nr_topics=TARGET_N_TOPICS)

reduced_info = topic_model.get_topic_info()
n_outliers_reduced = (pd.Series(topic_model.topics_) == -1).sum()
print(f"outlier topic (-1): {n_outliers_reduced} / {len(topic_model.topics_)} docs "
      f"({n_outliers_reduced / len(topic_model.topics_) * 100:.1f}%)")
print(f"{(reduced_info['Topic'] != -1).sum()} topics after reduction (target was {TARGET_N_TOPICS})")
reduced_info.head(30)

In [ ]:
topic_model

In [ ]:
topic_info.to_csv("../data/processed/all_topics.csv", index = False)

In [ ]:
top_topics = topic_info[topic_info["Topic"] != -1].head(15)

fig, ax = plt.subplots(figsize=plot_style.FIGSIZE_TALL)
sns.barplot(data=top_topics, y="Name", x="Count", order=top_topics["Name"], ax=ax)
plot_style.annotate_bars(ax)
ax.set_title("Top 15 BERTopic Topics by Size")
ax.set_xlabel("Number of Posts")
ax.set_ylabel("Topic")
#plot_style.savefig(fig, "bertopic_top_topics")
plt.show()

In [ ]:
topics_over_time = topic_model.topics_over_time(topic_docs.tolist(), topic_dates.tolist(), nr_bins=40)

topic_names = topic_info.set_index("Topic")["Name"]
top_topic_ids = top_topics["Topic"].head(8).tolist()
plot_data = topics_over_time[topics_over_time["Topic"].isin(top_topic_ids)].copy()
plot_data["Topic Name"] = plot_data["Topic"].map(topic_names)

fig, ax = plt.subplots(figsize=plot_style.FIGSIZE_WIDE)
sns.lineplot(data=plot_data, x="Timestamp", y="Frequency", hue="Topic Name", ax=ax)
ax.set_title("Top Topics Over Time")
ax.set_xlabel("Date")
ax.set_ylabel("Posts per Time Bin")
ax.legend(title="Topic", bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
#plot_style.savefig(fig, "bertopic_topics_over_time")
plt.show()

In [ ]:
df["topic"] = pd.NA
df.loc[non_empty, "topic"] = topics
df["topic_name"] = df["topic"].map(topic_names)

df["topic_name"].value_counts(dropna=False)

### Cross-Check: Manual Categories vs. BERTopic Topics

Axis E (§6.3.1) combines BERTopic's discovered topics with this manual hashtag codebook. For each BERTopic topic, the most common manual category among its posts -- a quick read on whether the two agree, and where they diverge.

In [ ]:
df_topic_valid = df.dropna(subset=["topic"])
df_topic_valid = df_topic_valid[df_topic_valid["topic"] != -1]

topic_vs_category = (
    df_topic_valid.explode("hashtag_topics")
    .groupby("topic_name")["hashtag_topics"]
    .value_counts(normalize=True)
    .groupby(level=0)
    .head(1)
    .rename("share")
    .reset_index()
    .sort_values("share", ascending=False)
)
topic_vs_category


## Takeaway

**Text length & logistics heuristics**: your fear that posts are mostly thin protest-call logistics doesn't hold up. 30.5% of posts contain an explicit event-logistics marker (clock time, date, or "Treffpunkt"/"kommt alle"-style phrase), but those posts are actually **longer**, not shorter (median 59 words vs. 45 for the rest). 94.1% of posts have ≥10 words and 83.9% have ≥20 words after stripping hashtags/mentions — DPM framing coding looks viable on text length grounds alone.

**Hashtag co-occurrence**: confirms real, distinguishable sub-themes beyond generic climate branding — a democracy/anti-AfD cluster (`#demokratie` ↔ `#fckafd`) and a feminism cluster (`#feminism` ↔ `#intersectionalfeminism`) both stand out clearly once the near-universal `#fridaysforfuture` tag (82.3% of posts, essentially zero variance) is excluded.

**Topic modeling**: the first BERTopic run collapsed to one giant cluster because hashtag boilerplate dominated the embeddings (see §4's diagnosis). Revised with hashtag-free embedding text, smaller/explicit HDBSCAN clustering, and a seed topic list targeting climate/mobilization/feminism/democracy — rerun on the GPU server to see whether it now recovers the sub-themes the hashtag correlation analysis already points to.

The zero-shot/LLM-based DPM scoring step (§6.3) is deferred to a later stage per your steer.